### 0) imports

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit
from sklearn.linear_model import Lasso
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.metrics import r2_score
import shap
from sklearn.linear_model import ElasticNet
import optuna

### 1) Answer the questions from the introduction

#### What is leave-one-out? Provide limitations and strengths

LeaveOneOut (или LOO) — это простая перекрестная проверка. Каждый обучающий набор создается путем взятия всех образцов, кроме одного, причем тестовый набор — это сам исключенный образец. \
Преимущества:
- Эта процедура перекрестной проверки не приводит к большим потерям данных, поскольку из обучающего набора удаляется только один образец
- Максимальное использование данных

Недостатки:
- С точки зрения точности, метод часто приводит к высокой дисперсии в качестве оценки ошибки теста
- Огромные вычислительные данные
- Проблема утечки

#### How do Grid Search, Randomized Grid Search, and Bayesian optimization work?

Grid Search:
- Заранее определяем интересующие нас параметры и затем уже сам алгоритм строит из них сетку

Randomized Grid Search:
- Этот метод случайным образом выбирает точки из заданного интервала значений

Bayesian optimization work:
- Это метод, который учится на своих прошлых значениях, каждая следующая попытка основывается на результатах предыдущих.

#### Explain classification of feature selection methods

Filter Methods:
- Оценивают каждый признак независимо от модели. Они смотрят на статистические свойства данных (корреляция, вариативность)

Wrapper Methods:
- Они перебирают разные подмножества признаков, обучают модель и находят лучший набор именно для данной модели

Embedded Methods:
- Отбор происходит прямо во время обучения модели. Модель сама решает, какие признаки важны

#### Explain how Pearson and Chi2 work

- Pearson измеряет силу линейной зависимости между двумя непрерывными переменными (от -1 до 1)
- Chi2 проверяет, есть ли зависимость между признаком и целевым классом. Если распределение признака одинаково для всех классов, то признак бесполезен (независим).

#### Explain how Lasso works

К функции потерь (ошибке) модели добавляется штраф, равный сумме абсолютных значений коэффициентов, а так он совпадает с линейной регрессией

#### Explain what permutation significance is

Берем один признак и случайным образом перемешиваем (переставляем) его значения в тестовой выборке. Если после порчи признака точность модели рухнула — значит, признак был критически важен. Если точность почти не изменилась — признак бесполезен.

#### Become familiar with SHAP

Метод перебирает все возможные комбинации признаков и оценивает, как добавление конкретного признака $X$ меняет предсказание по сравнению со средним значением

### 2) Introduction

In [2]:
data = pd.read_json('../datasets/train.json')
data['created'] = pd.to_datetime(data['created'])
data.sort_values(by='created', ascending=True, inplace=True)
data.reset_index(inplace=True)
df = pd.DataFrame()

In [3]:
data = data[(data['bathrooms'] >= data['bathrooms'].quantile(0.01)) & (data['bathrooms'] <= data['bathrooms'].quantile(0.99))]
data = data[(data['bedrooms'] >= data['bedrooms'].quantile(0.01)) & (data['bedrooms'] <= data['bedrooms'].quantile(0.99))]
data = data[(data['price'] >= data['price'].quantile(0.01)) & (data['price'] <= data['price'].quantile(0.99))]

In [4]:
data['interest_level'] = data['interest_level'].map({'low':0, 'medium':1, 'high':2})

In [5]:
all_features = []
for index, row in data.iterrows():
    for feature in row['features']:
        features = feature.replace('[', '').replace(']', '').replace("'", '').replace('"', '').replace(' ', '')
        all_features.append(features)

In [6]:
all_features = Counter(all_features).most_common(20)
feature_list = [feature for feature, _ in all_features]

In [7]:
df['bathrooms'] = data['bathrooms']
df['bedrooms'] = data['bedrooms']
df['interest_level'] = data['interest_level']
for feature in feature_list:
    df[feature] = data['features'].apply(lambda x: 1 if feature in x else 0)

### 3) Implement the next methods

In [8]:
def random_split_train_test(data, test_size, seed=21):
    np.random.seed(seed)
    data = np.random.permutation(data)
    len_test = int(round(len(data) * test_size))
    train = data[len_test:]
    test = data[:len_test]
    return train, test

In [9]:
def random_split_train_valid_test(data, test_size, validation_size, seed=21):
    np.random.seed(seed)
    validation_size = validation_size / (1 - test_size)
    train_valid, test = random_split_train_test(data, test_size)
    train, valid = random_split_train_test(train_valid, validation_size)
    return train, valid, test

In [10]:
def split_train_test(data, date_column, date_split):
    data_sorted = data.sort_values(date_column)
    train_mask = data_sorted[date_column] < date_split
    test_mask = data_sorted[date_column] >= date_split
    train = data_sorted[train_mask]
    test = data_sorted[test_mask]
    return train, test

In [11]:
def split_train_valid_test(data, date_column, date_valid, date_test):
    data_sorted = data.sort_values(date_column)
    train = data_sorted[data_sorted[date_column] < date_valid]
    valid = data_sorted[(data_sorted[date_column] >= date_valid) & 
                        (data_sorted[date_column] < date_test)]
    test = data_sorted[data_sorted[date_column] >= date_test]
    return train, valid, test

### 4) Implement the next cross-validation methods

In [12]:
class MyKFold:
    def __init__(self, k=5):
        self.n_splits = k

    def _iter_test_indices(self, X):
        n_samples = len(X)
        indices = np.arange(n_samples)
        fold_sizes = np.full(self.n_splits, n_samples // self.n_splits, dtype=int)
        fold_sizes[:n_samples % self.n_splits] += 1
        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            yield indices[start:stop]
            current = stop
            
    def split(self, X):
        n_samples = len(X)
        indices = np.arange(n_samples)
        for test_index in self._iter_test_indices(X):
            train_index = np.setdiff1d(indices, test_index)
            yield train_index, test_index

In [13]:
class MyGroupKFold:
    def __init__(self, k=5):
        self.n_splits = k

    def _iter_test_indices(self, X, group_field):
        unique_groups = np.unique(group_field)
        fold_groups = np.array_split(unique_groups, self.n_splits)
        for fold_group in fold_groups:
            test_index = np.where(np.isin(group_field, fold_group))
            yield test_index
            
    def split(self, X, group_field=None):
        n_samples = len(X)
        indices = np.arange(n_samples)
        for test_index in self._iter_test_indices(X, group_field):
            train_index = np.setdiff1d(indices, test_index)
            yield train_index, test_index

In [14]:
class MyStratifiedKFold:
    def __init__(self, k=5):
        self.n_splits = k

    def _iter_test_indices(self, X, stratify_field):
        classes = np.unique(stratify_field)
        classes_indices = [np.where(stratify_field == c)[0] for c in classes]
        fold_classes = [np.array_split(idx, self.n_splits) for idx in classes_indices]
        for i in range(self.n_splits):
            test_index = np.concatenate([fold_class[i] for fold_class in fold_classes])
            yield test_index

    def split(self, X, stratify_field=None):
        n_samples = len(X)
        indices = np.arange(n_samples)
        for test_index in self._iter_test_indices(X, stratify_field):
            train_index = np.setdiff1d(indices, test_index)
            yield train_index, test_index

In [15]:
class MyTimeSeriesSplit:
    def __init__(self, k=5):
        self.n_splits = k

    def _iter_test_indices(self, X, date_field):
        date_field = np.array(date_field)
        sorted_indices = np.argsort(date_field)
        n_samples = len(X)
        test_size = n_samples // (self.n_splits + 1)
        
        for i in range(1, self.n_splits + 1):
            split_point = i * test_size
            test_end = split_point + test_size
            if i == self.n_splits:
                test_end = n_samples
            yield sorted_indices[split_point:test_end]

    def split(self, X, date_field=None):
        n_samples = len(X)
        date_field = np.array(date_field)
        sorted_indices = np.argsort(date_field)
        
        for test_index in self._iter_test_indices(X, date_field):
            first_test_pos = np.where(sorted_indices == test_index[0])[0][0]
            train_index = sorted_indices[:first_test_pos]
            
            yield train_index, test_index

### 5) Cross-validation comparison

In [16]:
X = df.copy()
y = data['interest_level']

In [17]:
my_kf = MyKFold()
for i, (train_index, test_index) in enumerate(my_kf.split(X)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Fold 0:
  Train: index=[ 9512  9513  9514 ... 47555 47556 47557]
  Test:  index=[   0    1    2 ... 9509 9510 9511]
Fold 1:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[ 9512  9513  9514 ... 19021 19022 19023]
Fold 2:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[19024 19025 19026 ... 28533 28534 28535]
Fold 3:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[28536 28537 28538 ... 38044 38045 38046]
Fold 4:
  Train: index=[    0     1     2 ... 38044 38045 38046]
  Test:  index=[38047 38048 38049 ... 47555 47556 47557]


In [18]:
sk_kf = KFold()
for i, (train_index, test_index) in enumerate(sk_kf.split(X)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Fold 0:
  Train: index=[ 9512  9513  9514 ... 47555 47556 47557]
  Test:  index=[   0    1    2 ... 9509 9510 9511]
Fold 1:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[ 9512  9513  9514 ... 19021 19022 19023]
Fold 2:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[19024 19025 19026 ... 28533 28534 28535]
Fold 3:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[28536 28537 28538 ... 38044 38045 38046]
Fold 4:
  Train: index=[    0     1     2 ... 38044 38045 38046]
  Test:  index=[38047 38048 38049 ... 47555 47556 47557]


In [19]:
groups = data['manager_id'].values

In [20]:
my_group_kfold = MyGroupKFold()
for i, (train_index, test_index) in enumerate(my_group_kfold.split(X, group_field=groups)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}, group={groups[train_index]}")
    print(f"  Test:  index={test_index}, group={groups[test_index]}")

Fold 0:
  Train: index=[    0     1     2 ... 47555 47556 47557], group=['f07272f8ceb99db4c1a7cbbd9ae7b75b' '3b630ec9cb6eee53b92cfac7f42e3bf4'
 '3b630ec9cb6eee53b92cfac7f42e3bf4' ... '634f618895493a04f7722f113a89947b'
 '699c325b818541f314b691b76f3238d7' '699c325b818541f314b691b76f3238d7']
  Test:  index=(array([    3,    13,    14, ..., 47548, 47552, 47553], shape=(9245,)),), group=['262471a6a678adb458e879b092b23b4b' '044573b60ce5da02f510e49f086707ec'
 '1a79c79b0366512deaa5053ae501964b' ... '25d4ea3f8ec14332bcf177e416c6747d'
 '0bafd514443193d057e8a60e45cb7ea6' '0bafd514443193d057e8a60e45cb7ea6']
Fold 1:
  Train: index=[    0     3     4 ... 47554 47556 47557], group=['f07272f8ceb99db4c1a7cbbd9ae7b75b' '262471a6a678adb458e879b092b23b4b'
 '7c5e4fc025b70c6540d6b0e06716b9dd' ... '6c2a16187e6855c132bb496b875a4ef7'
 '699c325b818541f314b691b76f3238d7' '699c325b818541f314b691b76f3238d7']
  Test:  index=(array([    1,     2,     5, ..., 47519, 47520, 47555], shape=(9618,)),), group=['3b630ec9cb

In [21]:
sk_group_kfold = GroupKFold()
for i, (train_index, test_index) in enumerate(sk_group_kfold.split(X, groups=groups)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}, group={groups[train_index]}")
    print(f"  Test:  index={test_index}, group={groups[test_index]}")

Fold 0:
  Train: index=[    0     1     2 ... 47555 47556 47557], group=['f07272f8ceb99db4c1a7cbbd9ae7b75b' '3b630ec9cb6eee53b92cfac7f42e3bf4'
 '3b630ec9cb6eee53b92cfac7f42e3bf4' ... '634f618895493a04f7722f113a89947b'
 '699c325b818541f314b691b76f3238d7' '699c325b818541f314b691b76f3238d7']
  Test:  index=[   12    16    18 ... 47524 47527 47550], group=['38e613cd90ba43943211be10168ee175' 'e6472c7237327dd3903b3d6f6a94515a'
 'e6472c7237327dd3903b3d6f6a94515a' ... 'babf967aeb47132007ac2dd76c204d49'
 'e6472c7237327dd3903b3d6f6a94515a' 'f5e44ef976867c828130d0e6e6cdbda5']
Fold 1:
  Train: index=[    0     4     5 ... 47553 47556 47557], group=['f07272f8ceb99db4c1a7cbbd9ae7b75b' '7c5e4fc025b70c6540d6b0e06716b9dd'
 '4ccafba721db54b92d97c0af85847c02' ... '0bafd514443193d057e8a60e45cb7ea6'
 '699c325b818541f314b691b76f3238d7' '699c325b818541f314b691b76f3238d7']
  Test:  index=[    1     2     3 ... 47548 47554 47555], group=['3b630ec9cb6eee53b92cfac7f42e3bf4' '3b630ec9cb6eee53b92cfac7f42e3bf4'
 '2

In [22]:
my_skf = MyStratifiedKFold()
for i, (train_index, test_index) in enumerate(my_skf.split(X, stratify_field=y)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Fold 0:
  Train: index=[ 9125  9131  9140 ... 47555 47556 47557]
  Test:  index=[   5    6    7 ... 9114 9117 9119]
Fold 1:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[ 9467  9469  9471 ... 18611 18612 18665]
Fold 2:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[19280 19281 19282 ... 27557 27563 27569]
Fold 3:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[28803 28804 28806 ... 37032 37034 37075]
Fold 4:
  Train: index=[    0     1     2 ... 38289 38290 38293]
  Test:  index=[38294 38295 38296 ... 47504 47512 47513]


In [23]:
sk_skf = StratifiedKFold()
for i, (train_index, test_index) in enumerate(sk_skf.split(X, y)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Fold 0:
  Train: index=[ 9125  9131  9140 ... 47555 47556 47557]
  Test:  index=[   0    1    2 ... 9700 9703 9707]
Fold 1:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[ 9125  9131  9140 ... 19273 19274 19275]
Fold 2:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[18592 18593 18597 ... 28800 28801 28802]
Fold 3:
  Train: index=[    0     1     2 ... 47555 47556 47557]
  Test:  index=[27578 27596 27610 ... 38289 38290 38293]
Fold 4:
  Train: index=[    0     1     2 ... 38289 38290 38293]
  Test:  index=[37115 37120 37130 ... 47555 47556 47557]


In [24]:
my_tscv = MyTimeSeriesSplit()
for i, (train_index, test_index) in enumerate(my_tscv.split(X, date_field=data['created'])):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Fold 0:
  Train: index=[   0    1    2 ... 7923 7924 7925]
  Test:  index=[ 7926  7927  7928 ... 15849 15850 15851]
Fold 1:
  Train: index=[    0     1     2 ... 15849 15850 15851]
  Test:  index=[15852 15853 15854 ... 23775 23776 23777]
Fold 2:
  Train: index=[    0     1     2 ... 23775 23776 23777]
  Test:  index=[23778 23779 23780 ... 31701 31702 31703]
Fold 3:
  Train: index=[    0     1     2 ... 31701 31702 31703]
  Test:  index=[31704 31705 31706 ... 39627 39628 39629]
Fold 4:
  Train: index=[    0     1     2 ... 39627 39628 39629]
  Test:  index=[39630 39631 39632 ... 47555 47556 47557]


In [25]:
tscv = TimeSeriesSplit()
for i, (train_index, test_index) in enumerate(tscv.split(X, data['created'])):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Fold 0:
  Train: index=[   0    1    2 ... 7925 7926 7927]
  Test:  index=[ 7928  7929  7930 ... 15851 15852 15853]
Fold 1:
  Train: index=[    0     1     2 ... 15851 15852 15853]
  Test:  index=[15854 15855 15856 ... 23777 23778 23779]
Fold 2:
  Train: index=[    0     1     2 ... 23777 23778 23779]
  Test:  index=[23780 23781 23782 ... 31703 31704 31705]
Fold 3:
  Train: index=[    0     1     2 ... 31703 31704 31705]
  Test:  index=[31706 31707 31708 ... 39629 39630 39631]
Fold 4:
  Train: index=[    0     1     2 ... 39629 39630 39631]
  Test:  index=[39632 39633 39634 ... 47555 47556 47557]


Для нашего набора методы будут не сильно отличаться, поэтому в данном случае можно выбрать KFold, как лучший метод перекрестной проверки

### 6) Feature Selection

In [26]:
X_train, X_valid, X_test = random_split_train_valid_test(X, 0.2, 0.2)
y_train, y_valid, y_test = random_split_train_valid_test(data['price'], 0.2, 0.2)

In [27]:
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

In [28]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [29]:
features = pd.DataFrame({'Name': df.columns,'Importance': np.abs(lasso.coef_)})
features.sort_values(by='Importance', inplace=True, ascending=False)
features

,Name,Importance
0,bathrooms,3055.740653
1,bedrooms,1797.801685
2,interest_level,859.167642
7,Doorman,573.872676
8,Dishwasher,161.790838
22,Terrace,118.120670
3,Elevator,104.287461
12,Pre-War,55.825789
18,Balcony,19.745967
4,HardwoodFloors,0.000000


In [30]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'valid', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'valid', 'test'])
result_R2 = pd.DataFrame(columns=['model', 'train', 'valid', 'test'])

In [31]:
y_pred_train = lasso.predict(X_train_scaled)
y_pred_valid = lasso.predict(X_valid_scaled)
y_pred_test = lasso.predict(X_test_scaled)

In [32]:
X_train_scaled_top_10_features = pd.DataFrame()
X_valid_scaled_top_10_features = pd.DataFrame()
X_test_scaled_top_10_features = pd.DataFrame()
indices = features.head(10).index
X_train_scaled_top_10_features = X_train_scaled[:,indices]
X_valid_scaled_top_10_features = X_valid_scaled[:,indices]
X_test_scaled_top_10_features = X_test_scaled[:,indices]

In [33]:
lasso.fit(X_train_scaled_top_10_features, y_train)
y_pred_train_top_10_features = lasso.predict(X_train_scaled_top_10_features)
y_pred_valid_top_10_features = lasso.predict(X_valid_scaled_top_10_features)
y_pred_test_top_10_features = lasso.predict(X_test_scaled_top_10_features)

In [34]:
result_MAE.loc[len(result_MAE)] = ['Lasso MinMaxScaler', mae(y_train, y_pred_train), mae(y_valid, y_pred_valid), mae(y_test, y_pred_test)]
result_RMSE.loc[len(result_RMSE)] = ['Lasso MinMaxScaler', rmse(y_train, y_pred_train), rmse(y_valid, y_pred_valid), rmse(y_test, y_pred_test)]
result_R2.loc[len(result_R2)] = ['Lasso MinMaxScaler', r2_score(y_train, y_pred_train), r2_score(y_valid, y_pred_valid), r2_score(y_test, y_pred_test)]

In [35]:
result_MAE.loc[len(result_MAE)] = ['Lasso top10 MinMaxScaler', mae(y_train, y_pred_train_top_10_features), mae(y_valid, y_pred_valid_top_10_features), mae(y_test, y_pred_test_top_10_features)]
result_RMSE.loc[len(result_RMSE)] = ['Lasso top10 MinMaxScaler', rmse(y_train, y_pred_train_top_10_features), rmse(y_valid, y_pred_valid_top_10_features), rmse(y_test, y_pred_test_top_10_features)]
result_R2.loc[len(result_R2)] = ['Lasso top10 MinMaxScaler', r2_score(y_train, y_pred_train_top_10_features), r2_score(y_valid, y_pred_valid_top_10_features), r2_score(y_test, y_pred_test_top_10_features)]

In [36]:
result_MAE

,model,train,valid,test
0,Lasso MinMaxScaler,668.672518,658.821751,665.124980
1,Lasso top10 MinMaxScaler,668.672709,658.822143,665.125393


In [37]:
result_RMSE

,model,train,valid,test
0,Lasso MinMaxScaler,939.338320,914.957220,932.435435
1,Lasso top10 MinMaxScaler,939.338309,914.957361,932.435674


In [38]:
result_R2

,model,train,valid,test
0,Lasso MinMaxScaler,0.57827,0.580479,0.594119
1,Lasso top10 MinMaxScaler,0.57827,0.580478,0.594118


In [39]:
def feature_selection(X, y):
    data = X.loc[:, X.std() > 0]
    corr = data.corrwith(y).abs()
    nan = data.isnull().mean()
    final_score = (1 - corr) + nan
    names_top_10 = final_score.sort_values().head(10).index.to_list()
    indices = [list(X.columns).index(name) for name in names_top_10]
    return indices

In [40]:
indices = feature_selection(X, data['price'])

In [41]:
X_train_scaled_top_10_features = X_train_scaled[:,indices]
X_valid_scaled_top_10_features = X_valid_scaled[:,indices]
X_test_scaled_top_10_features = X_test_scaled[:,indices]

In [42]:
lasso.fit(X_train_scaled_top_10_features, y_train)
y_pred_train_top_10_features = lasso.predict(X_train_scaled_top_10_features)
y_pred_valid_top_10_features = lasso.predict(X_valid_scaled_top_10_features)
y_pred_test_top_10_features = lasso.predict(X_test_scaled_top_10_features)

In [43]:
result_MAE.loc[len(result_MAE)] = ['Lasso top10_selection MinMaxScaler', mae(y_train, y_pred_train_top_10_features), mae(y_valid, y_pred_valid_top_10_features), mae(y_test, y_pred_test_top_10_features)]
result_RMSE.loc[len(result_RMSE)] = ['Lasso top10_selection MinMaxScaler', rmse(y_train, y_pred_train_top_10_features), rmse(y_valid, y_pred_valid_top_10_features), rmse(y_test, y_pred_test_top_10_features)]
result_R2.loc[len(result_R2)] = ['Lasso top10_selection MinMaxScaler', r2_score(y_train, y_pred_train_top_10_features), r2_score(y_valid, y_pred_valid_top_10_features), r2_score(y_test, y_pred_test_top_10_features)]

In [44]:
result_MAE

,model,train,valid,test
0,Lasso MinMaxScaler,668.672518,658.821751,665.124980
1,Lasso top10 MinMaxScaler,668.672709,658.822143,665.125393
2,Lasso top10_selection MinMaxScaler,668.672775,658.822322,665.125238


In [45]:
result_RMSE

,model,train,valid,test
0,Lasso MinMaxScaler,939.338320,914.957220,932.435435
1,Lasso top10 MinMaxScaler,939.338309,914.957361,932.435674
2,Lasso top10_selection MinMaxScaler,939.338320,914.957404,932.435501


In [46]:
result_R2

,model,train,valid,test
0,Lasso MinMaxScaler,0.57827,0.580479,0.594119
1,Lasso top10 MinMaxScaler,0.57827,0.580478,0.594118
2,Lasso top10_selection MinMaxScaler,0.57827,0.580478,0.594119


In [47]:
def my_permutation_importance(model, X, y, seed=21):
    base_score = r2_score(y, model.predict(X))
    importances = []
    for i in range(X.shape[1]):
        X_copy = X.copy()
        np.random.seed(seed)
        np.random.shuffle(X_copy[:, i])
        new_score = r2_score(y, model.predict(X_copy))
        importances.append(base_score - new_score)
    top_10_indices = np.argsort(importances)[-10:][::-1]
    return top_10_indices

In [48]:
lasso.fit(X_train_scaled, y_train)
indices = my_permutation_importance(lasso, X_train_scaled, y_train)

In [49]:
X_train_scaled_top_10_features = X_train_scaled[:,indices]
X_valid_scaled_top_10_features = X_valid_scaled[:,indices]
X_test_scaled_top_10_features = X_test_scaled[:,indices]

In [50]:
lasso.fit(X_train_scaled_top_10_features, y_train)
y_pred_train_top_10_features = lasso.predict(X_train_scaled_top_10_features)
y_pred_valid_top_10_features = lasso.predict(X_valid_scaled_top_10_features)
y_pred_test_top_10_features = lasso.predict(X_test_scaled_top_10_features)

In [51]:
result_MAE.loc[len(result_MAE)] = ['Lasso permutation MinMaxScaler', mae(y_train, y_pred_train_top_10_features), mae(y_valid, y_pred_valid_top_10_features), mae(y_test, y_pred_test_top_10_features)]
result_RMSE.loc[len(result_RMSE)] = ['Lasso permutation MinMaxScaler', rmse(y_train, y_pred_train_top_10_features), rmse(y_valid, y_pred_valid_top_10_features), rmse(y_test, y_pred_test_top_10_features)]
result_R2.loc[len(result_R2)] = ['Lasso permutation MinMaxScaler', r2_score(y_train, y_pred_train_top_10_features), r2_score(y_valid, y_pred_valid_top_10_features), r2_score(y_test, y_pred_test_top_10_features)]

In [52]:
result_MAE

,model,train,valid,test
0,Lasso MinMaxScaler,668.672518,658.821751,665.124980
1,Lasso top10 MinMaxScaler,668.672709,658.822143,665.125393
2,Lasso top10_selection MinMaxScaler,668.672775,658.822322,665.125238
3,Lasso permutation MinMaxScaler,668.673573,658.823407,665.126185


In [53]:
result_RMSE

,model,train,valid,test
0,Lasso MinMaxScaler,939.338320,914.957220,932.435435
1,Lasso top10 MinMaxScaler,939.338309,914.957361,932.435674
2,Lasso top10_selection MinMaxScaler,939.338320,914.957404,932.435501
3,Lasso permutation MinMaxScaler,939.338313,914.957407,932.435820


In [54]:
result_R2

,model,train,valid,test
0,Lasso MinMaxScaler,0.57827,0.580479,0.594119
1,Lasso top10 MinMaxScaler,0.57827,0.580478,0.594118
2,Lasso top10_selection MinMaxScaler,0.57827,0.580478,0.594119
3,Lasso permutation MinMaxScaler,0.57827,0.580478,0.594118


In [55]:
lasso.fit(X_train_scaled, y_train)
explainer = shap.LinearExplainer(lasso, X_train_scaled)

In [56]:
shap_values = explainer.shap_values(X_train_scaled)
shap_values = np.abs(shap_values).mean(axis=0)

In [57]:
shap_table = pd.DataFrame({'features':df.columns, 'shap_value':shap_values}) 
shap_table.sort_values(by='shap_value', inplace=True, ascending=False)
shap_table

,features,shap_value
0,bathrooms,449.719908
1,bedrooms,404.962170
7,Doorman,272.528582
2,interest_level,226.151327
8,Dishwasher,80.359027
3,Elevator,52.282761
12,Pre-War,17.789789
22,Terrace,9.595866
18,Balcony,2.391564
4,HardwoodFloors,0.000000


In [58]:
indices = shap_table.head(10).index
X_train_scaled_top_10_features = X_train_scaled[:,indices]
X_valid_scaled_top_10_features = X_valid_scaled[:,indices]
X_test_scaled_top_10_features = X_test_scaled[:,indices]

In [59]:
lasso.fit(X_train_scaled_top_10_features, y_train)
y_pred_train_top_10_features = lasso.predict(X_train_scaled_top_10_features)
y_pred_valid_top_10_features = lasso.predict(X_valid_scaled_top_10_features)
y_pred_test_top_10_features = lasso.predict(X_test_scaled_top_10_features)

In [60]:
result_MAE.loc[len(result_MAE)] = ['Lasso shap MinMaxScaler', mae(y_train, y_pred_train_top_10_features), mae(y_valid, y_pred_valid_top_10_features), mae(y_test, y_pred_test_top_10_features)]
result_RMSE.loc[len(result_RMSE)] = ['Lasso shap MinMaxScaler', rmse(y_train, y_pred_train_top_10_features), rmse(y_valid, y_pred_valid_top_10_features), rmse(y_test, y_pred_test_top_10_features)]
result_R2.loc[len(result_R2)] = ['Lasso shap MinMaxScaler', r2_score(y_train, y_pred_train_top_10_features), r2_score(y_valid, y_pred_valid_top_10_features), r2_score(y_test, y_pred_test_top_10_features)]

In [61]:
result_MAE

,model,train,valid,test
0,Lasso MinMaxScaler,668.672518,658.821751,665.124980
1,Lasso top10 MinMaxScaler,668.672709,658.822143,665.125393
2,Lasso top10_selection MinMaxScaler,668.672775,658.822322,665.125238
3,Lasso permutation MinMaxScaler,668.673573,658.823407,665.126185
4,Lasso shap MinMaxScaler,668.673568,658.823399,665.126180


In [62]:
result_RMSE

,model,train,valid,test
0,Lasso MinMaxScaler,939.338320,914.957220,932.435435
1,Lasso top10 MinMaxScaler,939.338309,914.957361,932.435674
2,Lasso top10_selection MinMaxScaler,939.338320,914.957404,932.435501
3,Lasso permutation MinMaxScaler,939.338313,914.957407,932.435820
4,Lasso shap MinMaxScaler,939.338313,914.957404,932.435822


In [63]:
result_R2

,model,train,valid,test
0,Lasso MinMaxScaler,0.57827,0.580479,0.594119
1,Lasso top10 MinMaxScaler,0.57827,0.580478,0.594118
2,Lasso top10_selection MinMaxScaler,0.57827,0.580478,0.594119
3,Lasso permutation MinMaxScaler,0.57827,0.580478,0.594118
4,Lasso shap MinMaxScaler,0.57827,0.580478,0.594118


### 7) Hyperparameter optimization

In [64]:
class MyGridSearchCV:
    def __init__(self, param_grid, cv):
        self.param_grid = param_grid
        self.cv = cv
        self.best_score = -float('inf')
        self.best_alpha = None
        self.best_l1_ratio = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        kf = MyKFold(k=self.cv)

        for alpha in self.param_grid[0]:
            for l1_ratio in self.param_grid[1]:
                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)

                scores = []
                for train_index, test_index in kf.split(X):
                    X_train, X_test = X[train_index], X[test_index]
                    y_train, y_test = y[train_index], y[test_index]

                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)

                    scores.append(r2_score(y_test, y_pred))

                avg_score = np.mean(scores)

                if avg_score > self.best_score:
                    self.best_score = avg_score
                    self.best_alpha = alpha
                    self.best_l1_ratio = l1_ratio
                    
    def best_params_(self):
        return self.best_alpha, self.best_l1_ratio

    def best_score_(self):
        return self.best_score

In [65]:
class MyRandomizedSearchCV(MyGridSearchCV):
    def __init__(self, param_grid, cv, n_iter):
        super().__init__(param_grid, cv)
        self.n_iter = n_iter

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        kf = MyKFold(k=self.cv)

        for i in range(self.n_iter):
            alpha = np.random.choice(self.param_grid[0])
            l1_ratio = np.random.choice(self.param_grid[1])

            model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
            scores = []
            
            for train_index, test_index in kf.split(X):
                X_train, X_test = X[train_index], X[test_index]
                y_train, y_test = y[train_index], y[test_index]

                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                scores.append(r2_score(y_test, y_pred))

            avg_score = np.mean(scores)

            if avg_score > self.best_score:
                self.best_score = avg_score
                self.best_alpha = alpha
                self.best_l1_ratio = l1_ratio

In [66]:
param_grid = [[0.01, 0.1, 1, 10], 
              [0.1, 0.3, 0.5, 0.7, 0.9]]

In [67]:
scaler.fit_transform(X)

array([[0.  , 0.25, 1.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.5 , ..., 0.  , 0.  , 0.  ],
       [0.5 , 0.75, 1.  , ..., 0.  , 0.  , 0.  ],
       ...,
       [0.  , 0.25, 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.75, 0.5 , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.75, 0.5 , ..., 0.  , 0.  , 0.  ]], shape=(47558, 23))

In [68]:
my_grid_search = MyGridSearchCV(param_grid, 5)
my_grid_search.fit(X, data['price'])
alpha, l1_ratio = my_grid_search.best_params_()
print(my_grid_search.best_score_())

0.580941146604333


In [69]:
elastic = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
elastic.fit(X_train_scaled, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",0.01
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.9
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [70]:
y_pred_train = elastic.predict(X_train_scaled)
y_pred_valid = elastic.predict(X_valid_scaled)
y_pred_test = elastic.predict(X_test_scaled)

In [71]:
result_MAE.loc[len(result_MAE)] = ['ElasticNet GridSearch MinMaxScaler', mae(y_train, y_pred_train), mae(y_valid, y_pred_valid), mae(y_test, y_pred_test)]
result_RMSE.loc[len(result_RMSE)] = ['ElasticNet GridSearch MinMaxScaler', rmse(y_train, y_pred_train), rmse(y_valid, y_pred_valid), rmse(y_test, y_pred_test)]
result_R2.loc[len(result_R2)] = ['ElasticNet GridSearch MinMaxScaler', r2_score(y_train, y_pred_train), r2_score(y_valid, y_pred_valid), r2_score(y_test, y_pred_test)]

In [72]:
param_grid = [[0.0001, 0.001, 0.01, 0.1], 
              [0.1, 0.3, 0.5, 0.7, 0.9]]

In [73]:
my_random_search = MyRandomizedSearchCV(param_grid, 5, 5)
my_random_search.fit(X, data['price'])
alpha, l1_ratio = my_random_search.best_params_()
print(my_grid_search.best_score_())

0.580941146604333


In [74]:
elastic = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
elastic.fit(X_train_scaled, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",np.float64(0.001)
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",np.float64(0.7)
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [75]:
y_pred_train = elastic.predict(X_train_scaled)
y_pred_valid = elastic.predict(X_valid_scaled)
y_pred_test = elastic.predict(X_test_scaled)

In [76]:
result_MAE.loc[len(result_MAE)] = ['ElasticNet RandomizedSearch MinMaxScaler', mae(y_train, y_pred_train), mae(y_valid, y_pred_valid), mae(y_test, y_pred_test)]
result_RMSE.loc[len(result_RMSE)] = ['ElasticNet RandomizedSearch MinMaxScaler', rmse(y_train, y_pred_train), rmse(y_valid, y_pred_valid), rmse(y_test, y_pred_test)]
result_R2.loc[len(result_R2)] = ['ElasticNet RandomizedSearch MinMaxScaler', r2_score(y_train, y_pred_train), r2_score(y_valid, y_pred_valid), r2_score(y_test, y_pred_test)]

In [77]:
result_MAE

,model,train,valid,test
0,Lasso MinMaxScaler,668.672518,658.821751,665.124980
1,Lasso top10 MinMaxScaler,668.672709,658.822143,665.125393
2,Lasso top10_selection MinMaxScaler,668.672775,658.822322,665.125238
3,Lasso permutation MinMaxScaler,668.673573,658.823407,665.126185
4,Lasso shap MinMaxScaler,668.673568,658.823399,665.126180
5,ElasticNet GridSearch MinMaxScaler,668.851398,658.874139,665.441431
6,ElasticNet RandomizedSearch MinMaxScaler,668.740015,658.850723,665.233957


In [78]:
result_RMSE

,model,train,valid,test
0,Lasso MinMaxScaler,939.338320,914.957220,932.435435
1,Lasso top10 MinMaxScaler,939.338309,914.957361,932.435674
2,Lasso top10_selection MinMaxScaler,939.338320,914.957404,932.435501
3,Lasso permutation MinMaxScaler,939.338313,914.957407,932.435820
4,Lasso shap MinMaxScaler,939.338313,914.957404,932.435822
5,ElasticNet GridSearch MinMaxScaler,939.456763,914.789919,932.780360
6,ElasticNet RandomizedSearch MinMaxScaler,939.348698,914.879394,932.521517


In [79]:
result_R2

,model,train,valid,test
0,Lasso MinMaxScaler,0.578270,0.580479,0.594119
1,Lasso top10 MinMaxScaler,0.578270,0.580478,0.594118
2,Lasso top10_selection MinMaxScaler,0.578270,0.580478,0.594119
3,Lasso permutation MinMaxScaler,0.578270,0.580478,0.594118
4,Lasso shap MinMaxScaler,0.578270,0.580478,0.594118
5,ElasticNet GridSearch MinMaxScaler,0.578164,0.580632,0.593818
6,ElasticNet RandomizedSearch MinMaxScaler,0.578261,0.580550,0.594044


In [80]:
def objective(trial, X, y):
    param = {
        'alpha': trial.suggest_float('alpha', 1e-5, 10.0, log=True),
        'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0)
    }
    
    kf = KFold(n_splits=5)
    scores = []

    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model = ElasticNet(**param, random_state=42)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        scores.append(r2_score(y_test, y_pred))

    return np.mean(scores)

In [81]:
study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: objective(trial, X_train_scaled, y_train), n_trials=50)

print(f"Лучший R2 на кросс-валидации: {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")

[I 2025-12-25 19:47:40,255] A new study created in memory with name: no-name-f768b725-ef1a-42ef-98e4-feed174939c4
[I 2025-12-25 19:47:40,318] Trial 0 finished with value: 0.5773954026886846 and parameters: {'alpha': 0.0005324137197160206, 'l1_ratio': 0.06891016951447582}. Best is trial 0 with value: 0.5773954026886846.
[I 2025-12-25 19:47:40,354] Trial 1 finished with value: 0.5660790053590649 and parameters: {'alpha': 0.013847326889260481, 'l1_ratio': 0.046810794995678084}. Best is trial 0 with value: 0.5773954026886846.
[I 2025-12-25 19:47:40,395] Trial 2 finished with value: 0.47346523638058624 and parameters: {'alpha': 0.28869058756959054, 'l1_ratio': 0.761898794234539}. Best is trial 0 with value: 0.5773954026886846.
[I 2025-12-25 19:47:40,430] Trial 3 finished with value: 0.577407120190573 and parameters: {'alpha': 0.004059314183625207, 'l1_ratio': 0.9766016685352547}. Best is trial 3 with value: 0.577407120190573.
[I 2025-12-25 19:47:40,461] Trial 4 finished with value: 0.139771

Лучший R2 на кросс-валидации: 0.5774
Лучшие параметры: {'alpha': 0.0004495070850416437, 'l1_ratio': 0.6405834685284073}
